# Chapter 8 Lab: Frame Potential and Optimal Frame Design

This notebook accompanies **Chapter 8**. It builds the frame potential,
its gradient, and the projected-gradient-descent tight-frame constructor,
then works through all five lab exercises.

Run the setup cell first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline

def frame_potential(V):
    '''FP(V) = tr((V V^T)^2) for V of shape (n, m).'''
    S = V @ V.T
    return np.trace(S @ S)

def fp_gradient(V):
    '''Gradient of FP with respect to V: 4 S V.'''
    S = V @ V.T
    return 4 * S @ V

def normalize_columns(V):
    norms = np.linalg.norm(V, axis=0)
    return V / norms

def fit_tight_frame(n, m, num_steps=500, lr=0.05, seed=0, verbose=False, V_init=None):
    if V_init is not None:
        V = normalize_columns(V_init.copy())
    else:
        rng = np.random.default_rng(seed)
        V = rng.standard_normal((n, m))
        V = normalize_columns(V)

    history = [frame_potential(V)]
    for step in range(num_steps):
        grad = fp_gradient(V)
        V = V - lr * grad
        V = normalize_columns(V)
        history.append(frame_potential(V))
        if verbose and step % 100 == 0:
            print(f"step {step}: FP = {history[-1]:.6f}")

    return V, history

## Lab Exercise 8.1: Verifying the Gradient Formula

For a random $V$ of shape $(3,5)$, verify $\nabla\operatorname{FP}=4SV$ against a finite-difference approximation, confirming agreement to at least 4 decimal places.

In [ ]:
rng = np.random.default_rng(42)
V_test = rng.standard_normal((3, 5))

grad_analytic = None  # TODO: fp_gradient(V_test)

eps = 1e-6
grad_numeric = np.zeros_like(V_test)
for i in range(V_test.shape[0]):
    for j in range(V_test.shape[1]):
        V_plus = V_test.copy(); V_plus[i, j] += eps
        V_minus = V_test.copy(); V_minus[i, j] -= eps
        grad_numeric[i, j] = None  # TODO: (frame_potential(V_plus) - frame_potential(V_minus)) / (2*eps)

max_error = np.abs(grad_analytic - grad_numeric).max()
print(f"Max error between analytic and finite-difference gradient: {max_error:.2e}")
print(f"Agreement to at least 4 decimal places: {max_error < 1e-4}")

## Lab Exercise 8.2: Reproducing Known Tight Frames

Run `fit\_tight\_frame` for $(n,m)=(2,3),(2,4),(2,5),(2,6)$, confirm FP matches $m^2/n$ and frame is tight. Compare $(2,3)$ to the triangular frame.

In [ ]:
for m in [3, 4, 5, 6]:
    n = 2
    V_final, history = None, None  # TODO: fit_tight_frame(n, m, num_steps=1000, lr=0.03, seed=1)

    fp_final = history[-1]
    predicted = m**2 / n
    S_final = V_final @ V_final.T
    eigs = np.linalg.eigvalsh(S_final)
    is_tight = np.isclose(eigs.min(), eigs.max(), atol=1e-4)

    print(f"(n,m)=({n},{m}): FP_final={fp_final:.6f}, predicted={predicted:.6f}, "
          f"match={np.isclose(fp_final, predicted, atol=1e-4)}, tight={is_tight}")

V3, _ = fit_tight_frame(2, 3, num_steps=1000, lr=0.03, seed=1)
print("\n=== (n,m)=(2,3): pairwise angles ===")
for i, j in combinations(range(3), 2):
    cos_angle = np.dot(V3[:,i], V3[:,j])
    angle_deg = np.degrees(np.arccos(np.clip(cos_angle, -1, 1)))
    print(f"angle(v{i},v{j}) = {angle_deg:.2f} degrees")

*Your comparison: is the (2,3) gradient-descent solution the same configuration as the Chapter 1 triangular frame (up to rotation)?* (replace this text)

## Lab Exercise 8.3: Initialization Sensitivity

Initialize 4 vectors clustered around $(1,0)$ (Gaussian noise std=0.01); run gradient descent. Plot FP vs. iteration (semi-log). Compare convergence speed to random init.

In [ ]:
rng2 = np.random.default_rng(5)

base = np.tile(np.array([[1.0], [0.0]]), (1, 4))
noise = None  # TODO: rng2.standard_normal((2,4)) * 0.01
V_clustered_init = None  # TODO: base + noise

V_clustered, history_clustered = None, None  # TODO: fit_tight_frame(2, 4, num_steps=3000, lr=0.03, V_init=V_clustered_init)
V_random, history_random = None, None        # TODO: fit_tight_frame(2, 4, num_steps=3000, lr=0.03, seed=1)

plt.figure(figsize=(8,5))
plt.semilogy(history_clustered, label='clustered init')
plt.semilogy(history_random, label='random init')
plt.axhline(8.0, color='gray', linestyle='--', label='true minimum (m^2/n=8)')
plt.xlabel('iteration')
plt.ylabel('frame potential (log scale)')
plt.legend()
plt.title('Convergence speed: clustered vs. random initialization')
plt.show()

print(f"Clustered final FP: {history_clustered[-1]:.6f}")
print(f"Random final FP: {history_random[-1]:.6f}")

*Your observation: does the clustered start reach the same minimum, and how many more iterations does it need?* (replace this text)

## Lab Exercise 8.4: A Frame with No Symmetry

Run `fit\_tight\_frame` for $(n,m)=(3,7)$. Verify tight with $A=7/3$. Compute all 21 pairwise angles; are the vectors equiangular?

In [ ]:
V7, history7 = None, None  # TODO: fit_tight_frame(3, 7, num_steps=2000, lr=0.02, seed=2)

S7 = V7 @ V7.T
eigs7 = np.linalg.eigvalsh(S7)
print(f"Eigenvalues: {eigs7}")
print(f"Predicted A = 7/3 = {7/3:.4f}")
print(f"Tight: {np.isclose(eigs7.min(), eigs7.max(), atol=1e-3)}")

angles_deg = []
for i, j in combinations(range(7), 2):
    cos_angle = None  # TODO: np.dot(V7[:,i], V7[:,j])
    angle_deg = np.degrees(np.arccos(np.clip(cos_angle, -1, 1)))
    angles_deg.append(angle_deg)

angles_deg = np.array(angles_deg)
print(f"\nPairwise angles: min={angles_deg.min():.2f}, max={angles_deg.max():.2f}, "
      f"mean={angles_deg.mean():.2f}")
print(f"Equiangular (all angles equal)? {np.isclose(angles_deg.max(), angles_deg.min(), atol=1.0)}")

*Your conclusion: is this frame equiangular, or merely tight?* (replace this text)

## Lab Exercise 8.5 (Optional, Challenge): Multiple Restarts

Write `best\_of\_n\_restarts`. For $(n,m)=(2,4)$, compare final FPs across 20 independent runs. Report the fraction that fail to reach the global minimum within 500 steps.

In [ ]:
def best_of_n_restarts(n, m, num_restarts=10, num_steps=500, lr=0.05):
    best_V, best_history = None, None
    best_fp = np.inf
    for seed in range(num_restarts):
        V, history = None, None  # TODO: fit_tight_frame(n, m, num_steps=num_steps, lr=lr, seed=seed)
        if history[-1] < best_fp:
            best_fp = history[-1]
            best_V, best_history = V, history
    return best_V, best_history

final_fps = []
for seed in range(20):
    V, history = fit_tight_frame(2, 4, num_steps=500, lr=0.05, seed=seed)
    final_fps.append(history[-1])

final_fps = np.array(final_fps)
true_min = 8.0
n_failed = np.sum(~np.isclose(final_fps, true_min, atol=1e-2))

print(f"Final FPs across 20 runs: {final_fps.round(4)}")
print(f"True minimum: {true_min}")
print(f"Number of runs that did NOT reach the true minimum: {n_failed} / 20")

*Your suggestion for improving the fraction of successful runs (e.g. larger learning rate, momentum, more steps):* (replace this text)